In [31]:
def check_numeric_accuracy(sample_output_a_txt) -> bool:
    # Normalize commas
    text = sample_output_a_txt.replace(",", "")

    checks = {
        "Cleaned revenue (97,358.30)": "97358.3",
        "COD return rate (44.4)": "44.4",
        "Highest-risk segment return rate (54.5)": "54.5",
        "Duplicate reconciliation delta (2,501.90)": "2501.9"
    }

    all_passed = True

    for label, value in checks.items():
        if value in text:
            print(f"PASS: {label}")
        else:
            print(f"FAIL: {label}")
            all_passed = False

    # March and 20,318.90 must both be present
    if "March" in text and "20318.9" in text:
        print("PASS: March peak revenue (20,318.90)")
    else:
        print("FAIL: March peak revenue (20,318.90)")
        all_passed = False

    return all_passed

In [32]:
import json

findings = {
    "cleaned_total_revenue_inr": 97358.3,
    "raw_total_revenue_inr": 99860.2,
    "duplicate_reconciliation_delta_inr": 2501.9,
    "return_rate_by_payment": {
        "COD": 44.4,
        "CARD": 14.7,
        "UPI": 18.9
    },
    "highest_risk_segment": {
        "payment_method": "COD",
        "city_tier": 2,
        "return_rate_pct": 54.5
    },
    "true_peak_month": {
        "month": "2026-03",
        "revenue_inr": 20318.9
    },
    "outlier_inflated_month": {
        "month": "2026-01",
        "apparent_revenue_inr": 29582.1,
        "corrected_revenue_inr": 11637.1
    }
}

with open("findings.json", "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2, ensure_ascii=False)

print("findings.json created successfully!")

findings.json created successfully!


In [33]:
def generate_scr_narrative_offline(findings: dict) -> dict:
    """
    Fully deterministic offline fallback.
    Uses only the supplied findings and makes no network/API call.
    """

    narrative = f"""Situation

Cleaned revenue is ₹{findings["cleaned_total_revenue_inr"]}, compared with raw revenue of ₹{findings["raw_total_revenue_inr"]}. Return rates are COD at {findings["return_rate_by_payment"]["COD"]}%, CARD at {findings["return_rate_by_payment"]["CARD"]}%, and UPI at {findings["return_rate_by_payment"]["UPI"]}%.

Complication

The duplicate reconciliation delta is ₹{findings["duplicate_reconciliation_delta_inr"]}. The highest-risk segment is {findings["highest_risk_segment"]["payment_method"]} in city tier {findings["highest_risk_segment"]["city_tier"]}, with a return rate of {findings["highest_risk_segment"]["return_rate_pct"]}%. The apparent revenue for {findings["outlier_inflated_month"]["month"]} was ₹{findings["outlier_inflated_month"]["apparent_revenue_inr"]}, while corrected revenue was ₹{findings["outlier_inflated_month"]["corrected_revenue_inr"]}.

Resolution

Use the cleaned revenue of ₹{findings["cleaned_total_revenue_inr"]} for reporting. The true peak month was {findings["true_peak_month"]["month"]}, with revenue of ₹{findings["true_peak_month"]["revenue_inr"]}. Review the highest-risk segment as part of operational follow-up."""

    return {
        "status": "success",
        "narrative": narrative,
        "tokens": 0
    }

In [34]:
offline_result = generate_scr_narrative_offline(findings)

print(offline_result)

{'status': 'success', 'narrative': 'Situation\n\nCleaned revenue is ₹97358.3, compared with raw revenue of ₹99860.2. Return rates are COD at 44.4%, CARD at 14.7%, and UPI at 18.9%.\n\nComplication\n\nThe duplicate reconciliation delta is ₹2501.9. The highest-risk segment is COD in city tier 2, with a return rate of 54.5%. The apparent revenue for 2026-01 was ₹29582.1, while corrected revenue was ₹11637.1.\n\nResolution\n\nUse the cleaned revenue of ₹97358.3 for reporting. The true peak month was 2026-03, with revenue of ₹20318.9. Review the highest-risk segment as part of operational follow-up.', 'tokens': 0}


In [45]:
from datetime import datetime

month = "2026-03"
month_name = datetime.strptime(month, "%Y-%m").strftime("%B")

print(month_name)

March


In [47]:
offline_result["narrative"] = offline_result["narrative"].replace(
    "2026-03", month_name
)

print(offline_result["narrative"])

Situation

Cleaned revenue is ₹97358.3, compared with raw revenue of ₹99860.2. Return rates are COD at 44.4%, CARD at 14.7%, and UPI at 18.9%.

Complication

The duplicate reconciliation delta is ₹2501.9. The highest-risk segment is COD in city tier 2, with a return rate of 54.5%. The apparent revenue for 2026-01 was ₹29582.1, while corrected revenue was ₹11637.1.

Resolution

Use the cleaned revenue of ₹97358.3 for reporting. The true peak month was March, with revenue of ₹20318.9. Review the highest-risk segment as part of operational follow-up.


In [48]:
result = check_numeric_accuracy(offline_result["narrative"])

print("\nOverall:", "PASS" if result else "FAIL")

PASS: Cleaned revenue (97,358.30)
PASS: COD return rate (44.4)
PASS: Highest-risk segment return rate (54.5)
PASS: Duplicate reconciliation delta (2,501.90)
PASS: March peak revenue (20,318.90)

Overall: PASS
